# Barkley pipe flow - Phase 2: survival statistics

Built on the discrete model, these reproduce the *statistical* signatures of pipe
transition: memoryless puff lifetimes $P(n)\sim e^{-n/\tau(R)}$, the decay/split
lifetime crossing near $R_\times\approx2040$, and the turbulence-fraction onset
near $R_c\approx2046$. Ensembles here are small for Colab; see
[`ROADMAP.md`](../ROADMAP.md) for the precision caveats.


In [ ]:
# --- environment setup (works locally and on Google Colab) ---
try:
    import barkley_pipe  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/yuvrajdabhi2212/barkley-pipe-flow.git"], check=True)
import matplotlib.pyplot as plt
import numpy as np

## Memoryless decay: survival $P(n)\sim e^{-n/\tau}$

In [ ]:
from barkley_pipe.statistics import sample_lifetimes, survival_function, fit_tau

for R, c in [(1800, "C0"), (1850, "C1"), (1900, "C2")]:
    life = sample_lifetimes(R, "decay", n_realizations=300, n_sites=200, max_steps=9000, seed=1)
    n, surv = survival_function(life); tau = fit_tau(life)
    plt.semilogy(n, surv, ".", color=c, ms=4, label=f"R={R} (tau~{tau:.0f})")
    nn = np.linspace(0, n.max(), 100); plt.semilogy(nn, np.exp(-nn/tau), "-", color=c, lw=1)
plt.xlabel("n (steps)"); plt.ylabel("P(n)"); plt.ylim(1e-3, 1.2); plt.legend(); plt.show()

Straight lines on the log axis = exponential (memoryless) survival.

## Decay vs splitting lifetimes: the $R_\times$ crossing (~1-2 min)

In [ ]:
from barkley_pipe.statistics import tau_of_R

Rd = np.array([1950, 2000, 2020, 2040, 2060])
Rs = np.array([2040, 2080, 2120, 2160, 2200])
_, td = tau_of_R(Rd, n_realizations=80, event="decay", n_sites=200, max_steps=9000)
_, ts = tau_of_R(Rs, n_realizations=80, event="splitting", n_sites=400, max_steps=8000, check_every=5)
plt.semilogy(Rd, td, "o-", label="tau_decay")
plt.semilogy(Rs, ts, "s-", label="tau_split")
plt.xlabel("R"); plt.ylabel("tau (steps)"); plt.legend()
plt.title("Decay and splitting lifetimes cross near R_x ~ 2040"); plt.show()

## Turbulence-fraction order parameter $F_t(R)$

In [ ]:
from barkley_pipe.discrete import DiscreteParams, initial_puff, simulate_discrete
from barkley_pipe.statistics import turbulence_fraction

Rs = np.array([2000, 2080, 2160, 2300, 2500])
ft = []
for R in Rs:
    q0, u0 = initial_puff(800, width=400, seed=0)
    qst = simulate_discrete(q0, u0, DiscreteParams(R=R), n_steps=4000, store_every=20)
    ft.append(turbulence_fraction(qst, tail=0.4))
plt.plot(Rs, ft, "o-"); plt.axvline(2046, color="C2", ls="--", label="R_c~2046")
plt.xlabel("R"); plt.ylabel("F_t"); plt.legend(); plt.title("Onset of sustained turbulence"); plt.show()

**Note.** The exact $R_\times\approx2040$, $R_c\approx2046$, and the
directed-percolation exponent $\beta_{DP}\approx0.276$ require large lattices and
critical-region sampling (Barkley used ~4000 realizations on $\sim10^5$ sites).
With the reduced Colab ensembles above these are reproduced *within tolerance*,
not to published precision.
